# Predios para Conservación

> **Contexto de dominio:** [`docs/fuentes/predios_conservacion.md`](../../docs/fuentes/predios_conservacion.md)  
> **Bloque:** A | **Línea:** `predios_conservacion`  
> **Variable principal:** `ndvi` ()  
> **Modelos sugeridos:** RANDOM_FOREST, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/predios_conservacion.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "predios_conservacion"
VARIABLE = "ndvi"
UNIDAD = ""

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `predios_conservacion` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "predios_conservacion.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/predios_conservacion.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/predios_conservacion.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "ndvi": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `predios_conservacion` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_predios_conservacion.html",
       title="EDA — Predios para Conservación", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "ndvi", title="Predios para Conservación — ndvi ()")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "ndvi", period="month")
plt.show()

<!-- ENRICHMENT: predios_conservacion -->

## 3c. AHP para priorizacion de predios PSA — Costo de oportunidad

El **AHP (Analytic Hierarchy Process)** sopesa criterios ambientales y socioeconomicos para priorizar donde invertir recursos del 1% (Ley 99/1993 Art.111 y Ley 2320/2023):

| Criterio | Peso AHP | Por que |
|---|---|---|
| NDVI (vigor vegetal) | 35% | Mayor cobertura = menor intervencion requerida |
| TWI (indice topografico humedad) | 25% | Alta TWI = zona de recarga prioritaria |
| Costo de oportunidad ($/ha) | 20% | Menor costo = mayor eficiencia del PSA |
| Pendiente del terreno | 10% | Alta pendiente = riesgo erosion sin cobertura |
| Acceso a fuente de agua | 10% | Abastece acueducto municipal |

**Costo de oportunidad:** beneficio economico sacrificado al dejar la ganaderia/agricultura por conservacion. PSA eficiente: compensacion >= costo oportunidad.

**RBC (Relacion Beneficio-Costo):** RBC > 1 = el valor del servicio hidrico supera el costo del incentivo.

In [ ]:
# AHP para priorizacion de predios PSA
np.random.seed(17)
n_predios = 20

# Variables por predio (simuladas, normalizadas 0-1)
predios_df = pd.DataFrame({
    'predio_id': [f'PRD-{i+1:03d}' for i in range(n_predios)],
    'ndvi':           np.clip(np.random.normal(0.55, 0.18, n_predios), 0, 1),
    'twi':            np.clip(np.random.normal(0.5, 0.2, n_predios), 0, 1),
    'pendiente':      np.clip(np.random.normal(0.4, 0.2, n_predios), 0, 1),
    'acceso_agua':    np.random.uniform(0, 1, n_predios),
    'costo_oport_ha': np.random.uniform(300_000, 2_500_000, n_predios).round(-3),  # $/ha/ano
})

# Normalizar costo de oportunidad (inverso: menor costo = mayor puntaje)
co_min, co_max = predios_df['costo_oport_ha'].min(), predios_df['costo_oport_ha'].max()
predios_df['costo_norm'] = 1 - (predios_df['costo_oport_ha'] - co_min) / (co_max - co_min)

# Pesos AHP
pesos = {'ndvi': 0.35, 'twi': 0.25, 'costo_norm': 0.20, 'pendiente': 0.10, 'acceso_agua': 0.10}

# Puntaje AHP ponderado
predios_df['puntaje_ahp'] = sum(
    predios_df[col] * peso for col, peso in pesos.items())
predios_df['puntaje_ahp'] = predios_df['puntaje_ahp'].round(4)
predios_df['rango_ahp'] = predios_df['puntaje_ahp'].rank(ascending=False).astype(int)

# RBC simplificado: valor servicio hidrico / costo PSA
VALOR_AGUA_HA = 1_500_000  # $/ha/ano valor agua para acueducto
INCENTIVO_PSA = 600_000    # $/ha/ano incentivo tipico PSA (Decreto 1007/2018)
predios_df['rbc'] = (VALOR_AGUA_HA * predios_df['ndvi']) / predios_df['costo_oport_ha']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Puntaje AHP por predio
top10 = predios_df.nlargest(10, 'puntaje_ahp')
ax = axes[0]
bars = ax.barh(top10['predio_id'], top10['puntaje_ahp'],
               color=['#27ae60' if r <= 5 else '#f1c40f' for r in top10['rango_ahp']], alpha=0.85)
ax.set_xlabel('Puntaje AHP (0-1)')
ax.set_title('Top 10 Predios Priorizados — AHP\n(NDVI 35%, TWI 25%, Costo 20%...)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Panel B: Scatter NDVI vs Costo de oportunidad coloreado por RBC
ax = axes[1]
sc = ax.scatter(predios_df['costo_oport_ha']/1e6,
                predios_df['ndvi'],
                c=predios_df['rbc'], cmap='RdYlGn', s=60, alpha=0.8)
plt.colorbar(sc, ax=ax, label='RBC (valor/costo)')
ax.axhline(0.5, color='green', ls='--', lw=1)
ax.set_xlabel('Costo de oportunidad (M$/ha/ano)')
ax.set_ylabel('NDVI (vigor vegetal)')
ax.set_title('NDVI vs Costo oportunidad (color=RBC)\n(PSA eficiente: RBC > 1)', fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Priorizacion PSA — Ley 99/1993 Art.111 + Ley 2320/2023 + Decreto 1007/2018',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

n_rbc_positivo = (predios_df['rbc'] >= 1.0).sum()
print(f'Predios con RBC >= 1 (PSA eficiente): {n_rbc_positivo}/{n_predios}')
print(f'Costo oportunidad promedio: ${predios_df["costo_oport_ha"].mean()/1e6:.2f} M/ha/ano')
print('Top 3 predios priorizados:')
print(predios_df.nsmallest(3, 'rango_ahp')[['predio_id','puntaje_ahp','costo_oport_ha','rbc']])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["ndvi"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `ndvi` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["ndvi"], variable="ndvi")
if rep.empty:
    print("Sin normas colombianas registradas para 'ndvi'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["ndvi"], method="linear")
print(f"Faltantes antes: {df["ndvi"].isna().sum()} | "
      f"después: {df_clean["ndvi"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["ndvi"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: predios_conservacion -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Predios de conservación / GIS

- **GIS-driven:** indicadores de cobertura y conectividad ecológica — Moran's I antes de modelar.
- **No-modelado predictivo de series:** los predios cambian de estado en eventos discretos (compra, restauración).

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Predios para Conservación (`predios_conservacion`)
- **Variable analizada:** `ndvi` ()
- **Modelos ejecutados:** RANDOM_FOREST, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/predios_conservacion.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.